In [ ]:
import h5py
import numpy as np

with h5py.File('data/multifold.h5', 'r') as f:
    
    print("Top-level keys:", list(f.keys()))
    
    def print_structure(name, obj):
        print(name, "->", type(obj).__name__, 
              getattr(obj, 'shape', ''), 
              getattr(obj, 'dtype', ''))
    
    f.visititems(print_structure)

Top-level keys: ['df']
df -> Group  
df/axis0 -> Dataset (200,) |S25
df/axis1 -> Dataset (418014,) int64
df/block0_items -> Dataset (148,) |S21
df/block0_values -> Dataset (418014, 148) float32
df/block1_items -> Dataset (50,) |S25
df/block1_values -> Dataset (418014, 50) float64
df/block2_items -> Dataset (2,) |S15
df/block2_values -> Dataset (418014, 2) int32


Here I went through the multifold.h5 file and found that the top structure is that of a 'dataframe' that has been converted to an hdf5 format. As you can see, it has 8 subfolders. The first one named 'axis0' contains 200 strings and seems to be the names of the columns in the original dataframe. Then the 'axis1' folder contains around 400k integer values, so this seems to be the row elements. Then we have the 'block0_items', 'block1_items', 'block2_items' which are all strings and add up to 200(148+50+2), so they seem to be the different divided columns. The rest of the files are the corresponding 'block0_values', 'block1_values', 'block2_values' which are float or integer type values, so correspond to the row entries of the respective columns. So in summary, the original dataframe had 200 columns and around 400k rows, and it has been divided into 3 blocks of columns (148, 50, and 2 columns respectively) and stored in this hdf5 format.

In [ ]:
import h5py

with h5py.File('data/multifold.h5', 'r') as f:
    columns = [c.decode('utf-8') for c in f['df/axis0'][:]]

with open('columns1.txt', 'w') as out:
    for i, col in enumerate(columns):
        out.write(f"{i}: {col}\n")

In the columns1.txt,we can see that:
    In columns 0-23: We have physics quantities measured per event.
    In columns 24-126 & 128-199: We have the different types of weights per event.
    In column 127: We have seem to a metadata called 'target_dd', it might also instead be a weight, we cant tell yet.

Now doing the same thing to the other two files:

In [8]:
import h5py
import numpy as np

with h5py.File('data/multifold_sherpa.h5', 'r') as f:
    
    print("Top-level keys:", list(f.keys()))
    
    def print_structure(name, obj):
        print(name, "->", type(obj).__name__, 
              getattr(obj, 'shape', ''), 
              getattr(obj, 'dtype', ''))
    
    f.visititems(print_structure)

Top-level keys: ['df']
df -> Group  
df/axis0 -> Dataset (51,) |S23
df/axis1 -> Dataset (326430,) int64
df/block0_items -> Dataset (23,) |S12
df/block0_values -> Dataset (326430, 23) float32
df/block1_items -> Dataset (26,) |S23
df/block1_values -> Dataset (326430, 26) float64
df/block2_items -> Dataset (2,) |S15
df/block2_values -> Dataset (326430, 2) int32


In [10]:
import h5py

with h5py.File('data/multifold_sherpa.h5', 'r') as f:
    columns = [c.decode('utf-8') for c in f['df/axis0'][:]]

with open('columns2.txt', 'w') as out:
    for i, col in enumerate(columns):
        out.write(f"{i}: {col}\n")

In [12]:
import h5py
import numpy as np

with h5py.File('data/multifold_nonDY.h5', 'r') as f:
    
    print("Top-level keys:", list(f.keys()))
    
    def print_structure(name, obj):
        print(name, "->", type(obj).__name__, 
              getattr(obj, 'shape', ''), 
              getattr(obj, 'dtype', ''))
    
    f.visititems(print_structure)

Top-level keys: ['df']
df -> Group  
df/axis0 -> Dataset (26,) |S15
df/axis1 -> Dataset (433397,) int64
df/block0_items -> Dataset (24,) |S15
df/block0_values -> Dataset (433397, 24) float32
df/block1_items -> Dataset (2,) |S15
df/block1_values -> Dataset (433397, 2) int32


In [13]:
import h5py

with h5py.File('data/multifold_nonDY.h5', 'r') as f:
    columns = [c.decode('utf-8') for c in f['df/axis0'][:]]

with open('columns3.txt', 'w') as out:
    for i, col in enumerate(columns):
        out.write(f"{i}: {col}\n")

Now i will run a few dry tests and quality checks for the files and the data.

In [17]:
import sys
!{sys.executable} -m pip install tables

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.5-cp312-cp312-win_amd64.whl.metadata (39 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/6.5 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.5 MB 8.5 MB/s eta 0:00:01
   ----------- ---------------------------- 1.8/6.5 MB 7.2 MB/s eta 0:00:01
   ----------------- ---------------------- 2.9/6.5 MB 6.7 MB/s eta 0:00:01
   --------------------------- ------------ 4.5/6.5 MB 7.3 MB/s eta 0:00:01
   --------------------------------- ------ 5.5/6.5 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 6.5/6.5 MB 6.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------------------- -------------------- 1.6/3.1 MB 8.4 MB/s e


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
df1 = pd.read_hdf('data/multifold.h5')
print(df1.isnull().sum().sum())

0


In [22]:
import pandas as pd
df2 = pd.read_hdf('data/multifold_sherpa.h5')
print(df2.isnull().sum().sum())

0


In [23]:
import pandas as pd
df3 = pd.read_hdf('data/multifold_nonDY.h5')
print(df3.isnull().sum().sum())

0


So we have ensured that there are no null values in the dataset provided.

In [ ]:
print(df1['weights_nominal'].describe())

count    418014.000000
mean          0.004329
std           0.004246
min           0.000002
25%           0.001286
50%           0.003106
75%           0.006026
max           0.075667
Name: weights_nominal, dtype: float64


In [25]:
print(df2['weights_nominal'].describe())

count    326430.000000
mean          0.005564
std           0.004452
min           0.000005
25%           0.002296
50%           0.004624
75%           0.007654
max           0.076179
Name: weights_nominal, dtype: float64


In [26]:
print(df3['weights_nominal'].describe())

count    433397.000000
mean          0.004178
std           0.004088
min           0.000001
25%           0.001212
50%           0.003000
75%           0.005860
max           0.076544
Name: weights_nominal, dtype: float64


now just checking a few values randomly in just the multifold.h5 folder to see if we can find any interesting patterns. Also checking if we can determine what the 'target_dd' file contains.

In [27]:
print(df1[['pT_ll', 'eta_l1', 'weights_nominal', 'target_dd']].head(10))

        pT_ll    eta_l1  weights_nominal  target_dd
0  479.442780 -0.117443         0.003518   0.001732
1  274.524994  0.313321         0.003855   0.004238
2  462.713226  0.766387         0.002460   0.001283
3  215.157608  1.083798         0.006565   0.004298
4  222.458313 -0.635713         0.002630   0.003160
5  217.321198 -1.181789         0.002220   0.002967
6  317.852539 -0.844253         0.000448   0.000643
7  278.654480 -1.258115         0.011258   0.012797
8  207.509888 -1.450905         0.003909   0.002772
9  242.772400  2.017605         0.003679   0.002429


In [28]:
print(df1['target_dd'].describe())
print(df1['target_dd'].unique()[:20])  # see if truly continuous

count    418014.000000
mean          0.004276
std           0.003927
min           0.000001
25%           0.001358
50%           0.003212
75%           0.006034
max           0.060618
Name: target_dd, dtype: float64
[0.00173183 0.00423787 0.00128255 0.00429775 0.00316048 0.00296719
 0.00064305 0.01279749 0.00277204 0.00242873 0.00530887 0.00235469
 0.00185586 0.00607406 0.00056109 0.00112335 0.00025249 0.00196306
 0.01483733 0.00232585]


target_dd seems to be a kind of weight. it's values match that of the nominal weights closely, which leads us to believe it might be a weight. But we can't tell for certain as there is no documentation provided for it.

Checking if the common columns have the same values in the 3 files.

In [29]:
import pandas as pd

df1 = pd.read_hdf('data/multifold.h5')
df2 = pd.read_hdf('data/multifold_sherpa.h5')
df3 = pd.read_hdf('data/multifold_nonDY.h5')

common_obs = ['pT_ll', 'eta_l1', 'pT_trackj1', 'm_trackj1']

print("=== weights_nominal stats ===")
for name, df in [('nominal', df1), ('sherpa', df2), ('nonDY', df3)]:
    print(f"{name}: mean={df['weights_nominal'].mean():.6f}, std={df['weights_nominal'].std():.6f}")

print("\n=== observable stats (should differ — different event sets) ===")
for col in common_obs:
    print(f"\n{col}:")
    for name, df in [('nominal', df1), ('sherpa', df2), ('nonDY', df3)]:
        print(f"  {name}: mean={df[col].mean():.4f}, std={df[col].std():.4f}")

=== weights_nominal stats ===
nominal: mean=0.004329, std=0.004246
sherpa: mean=0.005564, std=0.004452
nonDY: mean=0.004178, std=0.004088

=== observable stats (should differ — different event sets) ===

pT_ll:
  nominal: mean=345.4685, std=170.8756
  sherpa: mean=316.6509, std=145.0953
  nonDY: mean=345.5620, std=170.7378

eta_l1:
  nominal: mean=0.0020, std=1.1231
  sherpa: mean=0.0029, std=1.1476
  nonDY: mean=0.0019, std=1.1209

pT_trackj1:
  nominal: mean=209.6384, std=166.3720
  sherpa: mean=211.5636, std=152.5353
  nonDY: mean=210.5833, std=167.1547

m_trackj1:
  nominal: mean=17.3114, std=13.4307
  sherpa: mean=16.0342, std=13.2414
  nonDY: mean=17.3044, std=13.5061
